In [1]:
import cv2
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from natsort import natsorted, index_natsorted

In [2]:
# 图片路径
image_dir = '/home/remot-ldl/DeepLearning/11-Paper3/03-Honey/data/Pictures'
# 获取所有jpg文件
image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]

# 方法1: RGB均值与标准差
def extract_rgb_features(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # 转换为RGB
    r_mean, g_mean, b_mean = np.mean(img, axis=(0, 1))
    r_std, g_std, b_std = np.std(img, axis=(0, 1))
    return [r_mean, g_mean, b_mean, r_std, g_std, b_std]

# 方法2: Lab色彩空间的均值
def extract_lab_features(image_path):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)  # 转换为Lab
    l_mean, a_mean, b_mean = np.mean(img, axis=(0, 1))
    return [l_mean, a_mean, b_mean]

# 方法3: L通道的颜色直方图
def extract_histogram_features(image_path, bins=8):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)  # 转换为Lab
    l_channel = img[:, :, 0]  # 只取L通道
    hist = cv2.calcHist([l_channel], [0], None, [bins], [0, 256])
    hist = hist.flatten()  # 转为1D数组
    return hist


In [5]:
# 存储所有特征
rgb_features = []
lab_features = []
hist_features = []
cleaned_index = []  # 存储处理后的索引（不含扩展名）
# 提取特征
for image_file in image_files:
    image_path = os.path.join(image_dir, image_file)

    # 提取文件名（不含扩展名）作为索引
    file_name_without_ext = os.path.splitext(image_file)[0]
    cleaned_index.append(file_name_without_ext)

    rgb_feat = extract_rgb_features(image_path)
    lab_feat = extract_lab_features(image_path)
    hist_feat = extract_histogram_features(image_path)
    
    rgb_features.append(rgb_feat)
    lab_features.append(lab_feat)
    hist_features.append(hist_feat)

# 创建DataFrame，并将文件名作为行索引
rgb_df = pd.DataFrame(rgb_features, columns=['r_mean', 'g_mean', 'b_mean', 'r_std', 'g_std', 'b_std'], index=cleaned_index)
rgb_df.index.name = 'Index'  # 设置索引名称

lab_df = pd.DataFrame(lab_features, columns=['l_mean', 'a_mean', 'b_mean'], index=cleaned_index)
lab_df.index.name = 'Index'  # 设置索引名称

hist_df = pd.DataFrame(hist_features, columns=[f'hist_{i+1}' for i in range(8)], index=cleaned_index)
hist_df.index.name = 'Index'  # 设置索引名称


# 使用natsorted对索引进行自然排序
rgb_df_sorted = rgb_df.reindex(index=natsorted(rgb_df.index))
lab_df_sorted = lab_df.reindex(index=natsorted(lab_df.index))
hist_df_sorted = hist_df.reindex(index=natsorted(hist_df.index))
print("使用natsort排序后的结果:")
print(rgb_df_sorted.head())


使用natsort排序后的结果:
           r_mean      g_mean     b_mean     r_std     g_std     b_std
Index                                                                 
E1     115.221680  104.936523  66.424805  2.147100  1.910312  1.748384
E2     114.828125  104.828125  69.828125  1.524459  1.524459  1.524459
E3     114.185547  109.185547  87.185547  1.410323  1.410323  1.410323
E4     114.936523  110.063477  87.561523  1.325410  1.358881  1.743886
E5     113.571289  106.571289  78.571289  1.586933  1.586933  1.586933


In [6]:
# 标准化
scaler = StandardScaler()
rgb_df_scaled = pd.DataFrame(scaler.fit_transform(rgb_df_sorted), columns=rgb_df_sorted.columns, index=rgb_df_sorted.index)
lab_df_scaled = pd.DataFrame(scaler.fit_transform(lab_df_sorted), columns=lab_df_sorted.columns, index=lab_df_sorted.index)
hist_df_scaled = pd.DataFrame(scaler.fit_transform(hist_df_sorted), columns=hist_df_sorted.columns, index=hist_df_sorted.index)

# 保存为csv文件
rgb_df_scaled.to_csv('rgb_features.csv')
lab_df_scaled.to_csv('lab_features.csv')
hist_df_scaled.to_csv('histogram_features.csv')

print("特征提取并保存成功！")


特征提取并保存成功！
